# Cross-country mention distribution

Compute a derivative dataset that counts how often articles from one country mention other countries (by name or well-known abbreviation) within the UAIR global article corpus. The end product is a per-country distribution of foreign mentions suitable for downstream analysis or visualization.


In [1]:
from __future__ import annotations

from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
import json
import re
import unicodedata

import pandas as pd
import pyarrow.parquet as pq
import pycountry
from tqdm.auto import tqdm


In [2]:
DATA_PATH = Path("/share/pierson/matt/UAIR/outputs/for_nov10workshop_global_results/classify/classify_all.parquet")
OUTPUT_DIR = Path("/share/pierson/matt/UAIR/outputs/derived")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "cross_country_mentions.parquet"

country_series = pq.read_table(DATA_PATH, columns=["country"]).to_pandas()["country"]
COUNTRY_CODES = sorted(country_series.dropna().str.upper().unique().tolist())
COUNTRY_SET = set(COUNTRY_CODES)

country_names = {code: pycountry.countries.get(alpha_2=code).name for code in COUNTRY_CODES}
codes_df = pd.DataFrame(
    [
        {"country_code": code, "country_name": country_names.get(code, code)}
        for code in COUNTRY_CODES
    ]
)

print(f"Loaded {len(COUNTRY_CODES)} source countries from {DATA_PATH}")
codes_df


Loaded 54 source countries from /share/pierson/matt/UAIR/outputs/for_nov10workshop_global_results/classify/classify_all.parquet


,country_code,country_name
0,AR,Argentina
1,AT,Austria
2,AU,Australia
3,BD,Bangladesh
4,BE,Belgium
5,BG,Bulgaria
6,CA,Canada
7,CH,Switzerland
8,CL,Chile
9,CO,Colombia


In [3]:
MANUAL_SYNONYMS: dict[str, list[str]] = {
    "CZ": ["czech republic"],
    "HK": ["hong kong", "hongkong"],
    "IR": ["iran", "islamic republic of iran"],
    "KR": ["south korea", "republic of korea"],
    "NL": ["holland", "the netherlands"],
    "SA": ["saudi arabia", "kingdom of saudi arabia", "ksa"],
    "SV": ["el salvador"],
    "TR": ["turkey", "republic of turkey", "republic of turkiye"],
    "TT": ["trinidad and tobago", "trinidad & tobago"],
    "TW": ["taiwan", "republic of china", "taiwan roc"],
    "TZ": ["tanzania", "united republic of tanzania"],
    "VE": ["venezuela", "bolivarian republic of venezuela"],
    "VN": ["vietnam", "viet nam"],
}


@dataclass(frozen=True)
class CountryAlias:
    country_code: str
    raw_alias: str
    normalized_alias: str
    kind: str


def normalize_text(value: str | None) -> str:
    """Lower-case, strip accents, and collapse whitespace."""

    if not isinstance(value, str):
        return ""
    text = unicodedata.normalize("NFKD", value)
    text = text.encode("ascii", "ignore").decode("ascii")
    text = text.lower()
    text = text.replace("-", " ")
    text = re.sub(r"[^a-z0-9\.\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def build_alias_records(country_codes: list[str]) -> list[CountryAlias]:
    aliases: list[CountryAlias] = []
    for code in country_codes:
        country = pycountry.countries.get(alpha_2=code)
        if country is None:
            continue
        candidates: list[tuple[str, str]] = [
            ("name", country.name),
            ("official", getattr(country, "official_name", "")),
            ("common", getattr(country, "common_name", "")),
            ("alpha3", country.alpha_3),
        ]
        candidates.extend(("manual", term) for term in MANUAL_SYNONYMS.get(code, []))
        for kind, term in candidates:
            if not term:
                continue
            normalized = normalize_text(term)
            if not normalized:
                continue
            aliases.append(CountryAlias(code, term, normalized, kind))
    # Drop duplicates by country+alias
    seen = set()
    unique_aliases: list[CountryAlias] = []
    for alias in aliases:
        key = (alias.country_code, alias.normalized_alias)
        if key in seen:
            continue
        seen.add(key)
        unique_aliases.append(alias)
    return unique_aliases



In [4]:
alias_records = build_alias_records(COUNTRY_CODES)
alias_df = pd.DataFrame(
    {
        "country_code": [r.country_code for r in alias_records],
        "raw_alias": [r.raw_alias for r in alias_records],
        "normalized_alias": [r.normalized_alias for r in alias_records],
        "alias_kind": [r.kind for r in alias_records],
    }
).sort_values(["country_code", "normalized_alias"]).reset_index(drop=True)

print(
    f"Prepared {len(alias_df)} unique aliases (mean {alias_df.groupby('country_code').size().mean():.1f} per country)"
)
alias_df.head(20)


Prepared 169 unique aliases (mean 3.1 per country)


,country_code,raw_alias,normalized_alias,alias_kind
0,AR,ARG,arg,alpha3
1,AR,Argentina,argentina,name
2,AR,Argentine Republic,argentine republic,official
3,AT,Austria,austria,name
4,AT,AUT,aut,alpha3
5,AT,Republic of Austria,republic of austria,official
6,AU,AUS,aus,alpha3
7,AU,Australia,australia,name
8,BD,Bangladesh,bangladesh,name
9,BD,BGD,bgd,alpha3


In [5]:
def compile_alias_patterns(alias_table: pd.DataFrame) -> list[dict[str, object]]:
    compiled: list[dict[str, object]] = []
    for row in alias_table.itertuples():
        escaped = re.escape(row.normalized_alias)
        pattern = re.compile(rf"(?<!\w){escaped}(?!\w)")
        compiled.append(
            {
                "country_code": row.country_code,
                "alias": row.normalized_alias,
                "pattern": pattern,
            }
        )
    return compiled


def count_country_mentions(normalized_text: str, compiled_aliases: list[dict[str, object]]) -> Counter[str]:
    counts: Counter[str] = Counter()
    if not normalized_text:
        return counts
    for entry in compiled_aliases:
        matches = entry["pattern"].findall(normalized_text)
        if matches:
            counts[entry["country_code"]] += len(matches)
    return counts


ALIAS_PATTERNS = compile_alias_patterns(alias_df)
len(ALIAS_PATTERNS)


169

In [6]:
sample_text = "The government of Canada met with officials from South Korea and Turkey to discuss trade."
normalized_sample = normalize_text(sample_text)
count_country_mentions(normalized_sample, ALIAS_PATTERNS)


Counter({'CA': 1, 'KR': 1, 'TR': 1})

In [7]:
pf = pq.ParquetFile(DATA_PATH)
occurrence_counts: dict[str, Counter[str]] = defaultdict(Counter)
article_hit_counts: dict[str, Counter[str]] = defaultdict(Counter)
articles_seen: Counter[str] = Counter()
articles_with_foreign_mentions: Counter[str] = Counter()

progress = tqdm(total=pf.metadata.num_rows, desc="Scanning articles", unit="articles")
for batch in pf.iter_batches(columns=["article_id", "country", "article_text"], batch_size=1000):
    df_batch = batch.to_pandas().dropna(subset=["country", "article_text"])
    if df_batch.empty:
        progress.update(0)
        continue
    df_batch["country"] = df_batch["country"].str.upper()
    df_batch = df_batch[df_batch["country"].isin(COUNTRY_SET)]
    progress.update(len(df_batch))
    for row in df_batch.itertuples(index=False):
        source = row.country
        articles_seen[source] += 1
        normalized = normalize_text(row.article_text)
        mention_counts = count_country_mentions(normalized, ALIAS_PATTERNS)
        mention_counts.pop(source, None)  # drop self-mentions
        if not mention_counts:
            continue
        articles_with_foreign_mentions[source] += 1
        for target, count in mention_counts.items():
            occurrence_counts[source][target] += int(count)
            article_hit_counts[source][target] += 1
progress.close()

print("Completed counting foreign mentions.")


Scanning articles:   0%|          | 0/386449 [00:00<?, ?articles/s]

Completed counting foreign mentions.


In [8]:
records: list[dict[str, object]] = []
for source, target_counts in occurrence_counts.items():
    for target, occurrences in target_counts.items():
        records.append(
            {
                "source_country": source,
                "mentioned_country": target,
                "mention_occurrences": int(occurrences),
                "article_hits": int(article_hit_counts[source][target]),
            }
        )

mentions_df = pd.DataFrame(records)
if mentions_df.empty:
    raise RuntimeError("No cross-country mentions were detected.")

mentions_df["source_total_mentions"] = (
    mentions_df.groupby("source_country")["mention_occurrences"].transform("sum")
)
mentions_df["mention_share"] = (
    mentions_df["mention_occurrences"] / mentions_df["source_total_mentions"]
)
mentions_df["source_articles_seen"] = mentions_df["source_country"].map(articles_seen)
mentions_df["source_articles_with_foreign_mentions"] = mentions_df["source_country"].map(
    articles_with_foreign_mentions
)
mentions_df["article_hit_share"] = (
    mentions_df["article_hits"] / mentions_df["source_articles_seen"].where(lambda x: x != 0, other=pd.NA)
)
mentions_df["source_country_name"] = mentions_df["source_country"].map(country_names)
mentions_df["mentioned_country_name"] = mentions_df["mentioned_country"].map(country_names)

mentions_df = mentions_df.sort_values(["source_country", "mention_occurrences"], ascending=[True, False])
mentions_df.head(20)


,source_country,mentioned_country,mention_occurrences,article_hits,source_total_mentions,mention_share,source_articles_seen,source_articles_with_foreign_mentions,article_hit_share,source_country_name,mentioned_country_name
2,AR,IL,2314,643,12830,0.180359,8457,3427,0.076032,Argentina,Israel
4,AR,CA,1417,823,12830,0.110444,8457,3427,0.097316,Argentina,Canada
15,AR,VE,1269,588,12830,0.098909,8457,3427,0.069528,Argentina,"Venezuela, Bolivarian Republic of"
0,AR,CL,1206,478,12830,0.093998,8457,3427,0.056521,Argentina,Chile
19,AR,CO,1190,457,12830,0.092751,8457,3427,0.054038,Argentina,Colombia
18,AR,UY,927,482,12830,0.072253,8457,3427,0.056994,Argentina,Uruguay
12,AR,IR,762,256,12830,0.059392,8457,3427,0.030271,Argentina,"Iran, Islamic Republic of"
9,AR,MA,668,367,12830,0.052065,8457,3427,0.043396,Argentina,Morocco
10,AR,FI,619,504,12830,0.048246,8457,3427,0.059596,Argentina,Finland
11,AR,PE,562,278,12830,0.043804,8457,3427,0.032872,Argentina,Peru


In [9]:
top_pairs = mentions_df.sort_values("mention_occurrences", ascending=False).head(20)
top_pairs[[
    "source_country",
    "mentioned_country",
    "mention_occurrences",
    "mention_share",
    "article_hits",
    "article_hit_share",
]]


,source_country,mentioned_country,mention_occurrences,mention_share,article_hits,article_hit_share
1290,IT,PE,12595,0.397143,4458,0.421242
1548,MA,CA,11315,0.459660,5486,0.472239
578,CO,CA,11202,0.222469,4942,0.503874
833,FR,CA,10608,0.449301,4679,0.764043
424,CL,CA,9683,0.546167,1203,0.154568
1184,IE,CA,9603,0.538859,3406,0.412748
579,CO,MY,8933,0.177408,4191,0.427304
331,CA,EE,8363,0.456471,67,0.007298
2665,VN,TH,8159,0.284593,1820,0.249657
2184,SG,CA,8021,0.319066,3214,0.380986


In [10]:
mention_matrix = mentions_df.pivot_table(
    index="source_country",
    columns="mentioned_country",
    values="mention_share",
    fill_value=0.0,
)
mention_matrix.round(3).head()


mentioned_country,AR,AT,AU,BD,BE,BG,CA,CH,CL,CO,...,SI,SV,TH,TR,TT,TW,TZ,UY,VE,VN
source_country,,,,,,,,,,,,,,,,,,,,,
AR,0.000,0.004,0.009,0.002,0.004,0.001,0.110,0.006,0.094,0.093,...,0.000,0.006,0.001,0.005,0.000,0.008,0.000,0.072,0.099,0.005
AT,0.010,0.000,0.046,0.005,0.011,0.009,0.379,0.020,0.007,0.008,...,0.007,0.001,0.008,0.012,0.001,0.003,0.003,0.003,0.003,0.008
AU,0.002,0.002,0.000,0.001,0.003,0.002,0.436,0.004,0.002,0.002,...,0.001,0.000,0.005,0.011,0.000,0.009,0.001,0.002,0.004,0.009
BD,0.005,0.003,0.018,0.000,0.009,0.000,0.261,0.012,0.002,0.006,...,0.002,0.000,0.013,0.012,0.000,0.004,0.000,0.002,0.002,0.018
BE,0.008,0.005,0.011,0.008,0.000,0.004,0.357,0.007,0.004,0.011,...,0.002,0.002,0.006,0.016,0.001,0.006,0.008,0.003,0.006,0.005


In [11]:
mentions_df.to_parquet(OUTPUT_PATH, index=False)
OUTPUT_PATH


PosixPath('/share/pierson/matt/UAIR/outputs/derived/cross_country_mentions.parquet')

The saved parquet file includes per-source-country totals, mention shares, and article hit rates for every destination country mentioned in the corpus. Use `mention_matrix` for quick heatmaps or load `mentions_df` for downstream modeling.
